## The goal
Produce a slim version of `books_with_interactions.parquet` that contains only the columns used at training time: `user_id`, `book_id`, `rating`, `data_split`.

The full file is ~19 GB because it duplicates every book attribute per interaction. The slim version drops all of that and uses zstd compression — typically lands at 200-500 MB, which fits comfortably in a 15 GB Drive quota.

Run once locally. Upload `training_interactions.parquet` to `data/transformed/` on Drive. Then `TrainingPairsDataset` will read this file on Colab instead of the full one.

In [1]:
import polars as pl
from mybookrec import DATA_DIR

In [2]:
# Stream the full parquet, project to the 4 columns we need, write with zstd compression.
# Using sink_parquet keeps the pipeline streaming end to end — no need to load 19 GB into RAM.
src = DATA_DIR / "transformed" / "books_with_interactions.parquet"
dst = DATA_DIR / "transformed" / "training_interactions.parquet"

(
    pl.scan_parquet(src)
    .select("user_id", "book_id", "rating", "data_split")
    .sink_parquet(dst, compression="zstd")
)

src_size = src.stat().st_size / 1e9
dst_size = dst.stat().st_size / 1e9
print(f"Source: {src.name}  {src_size:.2f} GB")
print(f"Slim:   {dst.name}  {dst_size:.2f} GB  ({dst_size / src_size:.1%} of original)")

Source: books_with_interactions.parquet  19.15 GB
Slim:   training_interactions.parquet  0.53 GB  (2.7% of original)


In [3]:
# Sanity check: row count and split distribution should match the source.
slim = pl.read_parquet(dst)
print(f"Rows: {len(slim):,}")
print(slim.head(3))
print()
print("Split distribution:")
print(slim.group_by("data_split").len().sort("data_split"))

Rows: 103,812,484
shape: (3, 4)
┌─────────────────────────────────┬──────────┬────────┬────────────┐
│ user_id                         ┆ book_id  ┆ rating ┆ data_split │
│ ---                             ┆ ---      ┆ ---    ┆ ---        │
│ str                             ┆ str      ┆ f64    ┆ str        │
╞═════════════════════════════════╪══════════╪════════╪════════════╡
│ 8842281e1d1347389f2ab93d60773d… ┆ 22078596 ┆ 4.0    ┆ validation │
│ 8842281e1d1347389f2ab93d60773d… ┆ 33232571 ┆ 5.0    ┆ validation │
│ 8842281e1d1347389f2ab93d60773d… ┆ 6644782  ┆ 4.0    ┆ validation │
└─────────────────────────────────┴──────────┴────────┴────────────┘

Split distribution:
shape: (3, 2)
┌────────────┬──────────┐
│ data_split ┆ len      │
│ ---        ┆ ---      │
│ str        ┆ u32      │
╞════════════╪══════════╡
│ test       ┆ 20628408 │
│ train      ┆ 72497704 │
│ validation ┆ 10686372 │
└────────────┴──────────┘
